In [1]:
import pandas as pd
import sklearn
import os
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, r2_score
import lightgbm as lgb
import xgboost as xgb

# Лимит RAM для CatBoost (подстрой под объём ОЗУ; при OOM уменьши до '4gb' или '3gb')
USED_RAM_LIMIT = '8gb'
# Количество потоков: -1 = все ядра; при OOM уменьши до 8 или 4
THREAD_COUNT = -1


In [2]:
df = pd.read_csv("dataset_combined.csv")

In [3]:
len(df)

124425

In [4]:
df.head(5)

,class,region,center_1500,wave_2002.46,wave_2001.50,wave_2000.54,wave_1999.58,wave_1998.62,wave_1997.66,wave_1996.70,...,wave_933.84,wave_932.67,wave_931.50,wave_930.32,wave_929.15,wave_927.98,wave_926.80,source_file,X,Y
0,endo,cortex,True,10178.839844,10165.766602,9905.018555,9887.439453,10146.862305,10076.990234,10103.795898,...,8520.928711,8504.200195,8459.643555,8528.562500,8428.372070,8516.528320,8544.746094,cortex_endo_1group_633nm_center1500_obj100_pow...,8315.753404,13149.765
1,endo,cortex,True,8581.288086,8638.810547,8600.557617,8402.886719,8733.147461,8773.278320,8492.099609,...,6477.447754,6480.410156,6346.338379,6430.672852,6350.172852,6303.942871,6360.422852,cortex_endo_1group_633nm_center1500_obj100_pow...,8317.753404,13149.765
2,endo,cortex,True,9784.027344,9825.862305,9821.366211,9712.325195,9629.458008,9617.164063,9691.076172,...,7805.496094,7752.506348,7866.547852,7751.487305,7670.717773,7921.655273,7821.629883,cortex_endo_1group_633nm_center1500_obj100_pow...,8319.753404,13149.765
3,endo,cortex,True,10320.031250,10461.222656,10365.109375,10535.625000,10562.352539,10356.543945,10393.744141,...,8771.543945,8848.993164,8930.694336,8686.974609,8734.430664,8784.007813,8869.933594,cortex_endo_1group_633nm_center1500_obj100_pow...,8321.753404,13149.765
4,endo,cortex,True,10223.289063,10265.124023,10192.575195,10159.259766,10431.695313,10241.587891,10383.295898,...,8349.567383,8525.615234,8418.961914,8415.105469,8469.037109,8640.638672,8634.600586,cortex_endo_1group_633nm_center1500_obj100_pow...,8323.753404,13149.765


## ⚠️ Проверка на data leakage (утечку по группам)

При случайном split по строкам строки из **одного source_file** попадают и в train, и в test. Спектры из одного файла (одного образца) почти идентичны — модель "подглядывает" в тесте. Ниже — диагностика.

In [5]:
# Диагностика: сколько source_file попадают и в train, и в test?
from sklearn.model_selection import train_test_split

x_tmp = df.drop(['class','source_file'], axis=1)
y_tmp = df['class']
x_tr, x_te, y_tr, y_te = train_test_split(x_tmp, y_tmp, test_size=0.2, random_state=42, shuffle=True)

train_files = set(df.loc[x_tr.index, 'source_file'])
test_files = set(df.loc[x_te.index, 'source_file'])
overlap = train_files & test_files

print(f"Уникальных source_file: {df['source_file'].nunique()}")
print(f"Файлов в train: {len(train_files)}, в test: {len(test_files)}")
print(f"Файлов в ОБОИХ (утечка!): {len(overlap)} ({100*len(overlap)/len(test_files):.1f}% теста)")

Уникальных source_file: 237
Файлов в train: 237, в test: 237
Файлов в ОБОИХ (утечка!): 237 (100.0% теста)


In [6]:
from sklearn.model_selection import GroupShuffleSplit

x = df.drop(['class','source_file'], axis=1)
y = df['class']
groups = df['source_file']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(x, y, groups))

x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [7]:
# Проверка: после split по группам утечки быть не должно
train_files_new = set(df.loc[train_idx, 'source_file'])
test_files_new = set(df.loc[test_idx, 'source_file'])
print("Файлов в train:", len(train_files_new), "| в test:", len(test_files_new))
print("Пересечение (должно быть 0):", len(train_files_new & test_files_new))

Файлов в train: 189 | в test: 48
Пересечение (должно быть 0): 0


In [8]:
# Закодируем region для XGB, GB, RF (они не поддерживают категориальные столбцы как CatBoost/LGBM)
x_train_enc = pd.get_dummies(x_train, columns=['region'])
x_test_enc = pd.get_dummies(x_test, columns=['region'])
# Выровняем столбцы теста под train (на случай отсутствующих категорий в test)
for c in x_train_enc.columns:
    if c not in x_test_enc.columns:
        x_test_enc[c] = 0
x_test_enc = x_test_enc[x_train_enc.columns]

# Метки для sklearn/XGB/LGBM: exo=0, endo=1, иначе 2
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)
# Соответствие: le.classes_ -> 0,1,2,...
print("Классы (метки):", dict(zip(le.classes_, range(len(le.classes_)))))

Классы (метки): {'control': 0, 'endo': 1, 'exo': 2}


In [9]:
# --- Модели для ансамбля ---
RANDOM_STATE = 42
N_ESTIMATORS = 300
MAX_DEPTH = 8

# Все модели обучаются на закодированных данных (region -> one-hot), чтобы избежать dtype object
# 1) LightGBM
model_lgbm = lgb.LGBMClassifier(
    n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, learning_rate=0.05,
    num_leaves=2**MAX_DEPTH, subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=THREAD_COUNT, verbose=-1,
)
model_lgbm.fit(
    x_train_enc, y_train_num,
    eval_set=[(x_test_enc, y_test_num)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
print("lgbm fit ended")
# 2) XGBoost
model_xgb = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
    n_jobs=THREAD_COUNT if THREAD_COUNT > 0 else None, use_label_encoder=False, eval_metric='mlogloss',
)
model_xgb.fit(
    x_train_enc, y_train_num,
    eval_set=[(x_test_enc, y_test_num)],
    verbose=False,
)
print("xgb fit ended")

# 3) CatBoost (без cat_features — все признаки числовые после get_dummies)
model_catboost = CatBoostClassifier(
    iterations=N_ESTIMATORS, depth=MAX_DEPTH, learning_rate=0.05,
    thread_count=THREAD_COUNT, used_ram_limit=USED_RAM_LIMIT,
    max_bin=128, bootstrap_type='Bernoulli', subsample=0.8,
    random_state=RANDOM_STATE, verbose=0,
)


model_catboost.fit(
    x_train_enc, y_train_num,
    eval_set=(x_test_enc, y_test_num),
    early_stopping_rounds=50,
    verbose=0,
)
print("catboost fit ended")
print("Все 3 модели обучены (LGBM, XGBoost, CatBoost).")

lgbm fit ended


C:\Users\alexk\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:36:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


xgb fit ended
catboost fit ended
Все 3 модели обучены (LGBM, XGBoost, CatBoost).


In [10]:
# Ансамбль: усреднение вероятностей (soft voting), затем argmax
def get_proba_lgbm(X):
    return model_lgbm.predict_proba(X)
def get_proba_xgb(X):
    return model_xgb.predict_proba(X)
def get_proba_catboost(X):
    return model_catboost.predict_proba(X)

# Предсказания вероятностей на тесте (все модели на x_test_enc)
p_lgbm = get_proba_lgbm(x_test_enc)
p_xgb  = get_proba_xgb(x_test_enc)
p_cat  = get_proba_catboost(x_test_enc)

# Усреднение и класс
proba_ensemble = (p_lgbm + p_xgb + p_cat) / 3
y_pred_ensemble = np.argmax(proba_ensemble, axis=1)

In [11]:
# Оценка каждой модели и ансамбля
def eval_model(name, y_pred):
    r2 = r2_score(y_test_num, y_pred)
    f1 = f1_score(y_test_num, y_pred, average='micro')
    print(f"{name:12}  R2: {r2:.4f}   F1(micro): {f1:.4f}")

y_pred_lgbm = model_lgbm.predict(x_test_enc)
y_pred_xgb  = model_xgb.predict(x_test_enc)
y_pred_cat  = model_catboost.predict(x_test_enc)

eval_model("LGBM",       y_pred_lgbm)
eval_model("XGBoost",    y_pred_xgb)
eval_model("CatBoost",   y_pred_cat)
eval_model("Ensemble",   y_pred_ensemble)

LGBM          R2: 0.9284   F1(micro): 0.9583
XGBoost       R2: 0.7250   F1(micro): 0.9583
CatBoost      R2: 0.7890   F1(micro): 0.8773
Ensemble      R2: 0.9273   F1(micro): 0.9577
